In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [4]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    tx = message.value
    if tx['amount'] > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Overwriting consumer_filter.py


In [5]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group', 
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Rozpoczynam wzbogacanie transakcji o poziom ryzyka...")

for message in consumer:
    tx = message.value
    
    if tx['amount'] > 3000:
        tx['risk_level'] = "HIGH"
    elif tx['amount'] > 1000:
        tx['risk_level'] = "MEDIUM"
    else:
        tx['risk_level'] = "LOW"
        
    print(f"ID: {tx['tx_id']} | Kwota: {tx['amount']:>8.2f} | RYZYKO: {tx['risk_level']}")

Writing consumer_enrich.py


In [6]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Rozpoczynam zbieranie statystyk per sklep...")

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    
    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0.0) + amount
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"\n--- PODSUMOWANIE (po {msg_count} wiadomościach) ---")
        print(f"{'Sklep':<12} | {'Liczba':<7} | {'Suma':<10} | {'Średnia':<8}")
        print("-" * 45)
        
        for s in sorted(store_counts.keys()):
            count = store_counts[s]
            suma = total_amount[s]
            srednia = suma / count
            print(f"{s:<12} | {count:<7} | {suma:>10.2f} | {srednia:>8.2f}")

Writing consumer_count.py


In [7]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {'count': 0, 'total': 0.0, 'min': float('inf'), 'max': 0.0})
msg_count = 0

print("Analiza kategorii...")

for message in consumer:
    tx = message.value
    cat = tx['category']
    amt = tx['amount']
    
    stats[cat]['count'] += 1
    stats[cat]['total'] += amt
    stats[cat]['min'] = min(stats[cat]['min'], amt)
    stats[cat]['max'] = max(stats[cat]['max'], amt)
    
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"\n--- Raport po {msg_count} transakcjach ---")
        for kategoria, s in stats.items():
            print(f"Kat: {kategoria} | Liczba: {s['count']} | Suma: {s['total']:.2f} | Min: {s['min']:.2f} | Max: {s['max']:.2f}")

Writing consumer_stats.py


In [8]:
# Zadanie 5.2 ODPOWIEDZI

# 1. Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?
# Parametr auto_offset_reset='earliest' sprawia, że po uruchomieniu konsumenta po zakończeniu producenta, przeczyta on wszystkie
# zapisane transakcje od samego początku pracy.

# 2. Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?
# Jeśli dwóch konsumentów miałoby tę samą group_id, to zamiast zjawiska wysyłania każdej transakcji do obu programów, wystąpiłoby
# rozdzielenie ich między tych konsumentów - część danych trafiłaby do pierwszego, a część do drugiego.
# Żaden z konsumentów nie zobaczyłby zatem pełnego strumienia danych, co mogłoby prowadzić do błednych wyników, szczególnie
# w kontekście filtrowania lub wyliczania statystyk. Różne group_id zapewniają niezależną pracę wszystkich konsumentów.

# 3. Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?
# Przetwarzanie bezstanowe oznacza, że program sprawdza niezależnie każdą transakcję, nie mając w pamięci tego, co działo się
# wcześniej. Na bieżąco sprawdza dany warunek, np. czy kwota danej transakcji przekracza 3000.
# Przetwarzanie stanowe różni się tym, że program utrzymuje w pamięci informacje o tym, co działo się wcześniej. Na podstawie
# zgromadzonych danych program generuje wynik końcowy, czyli na przykład średnie, sumy czy inne statystyki zagregowane.


In [9]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
import time
from collections import defaultdict

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='latest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_windows = defaultdict(list)

print("System wykrywania anomalii uruchomiony...")

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    now = time.time()
    
    user_windows[user_id].append(now)
    
    user_windows[user_id] = [t for t in user_windows[user_id] if now - t <= 60]
    
    if len(user_windows[user_id]) > 3:
        print(f"ANOMALIA: Uzytkownik {user_id} - ponad 3 transakcje w 60s.")

Writing consumer_anomaly.py
